In [ ]:
# Falure analyis for missing pin

> Starting from 20240521 Trying to make sure missing pin problem

In [ ]:
#| default_exp failure_analysis.missing_pin

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pathlib import Path
import numpy as np
from fastcore.all import *
from datetime import datetime, timedelta
from fastcore.imports import *  
import os

In [ ]:
import tensorflow as tf 

In [ ]:
path_et4 = Path(r'\\WARSDV513\aoi\Easy\ET4\Vision Orientation\Failed_images')
path_et4_outgonig = Path(r'\\WARSDV513\aoi\Easy\ET4\Vision Orientation\Failed_images_outgoing')
path_et3 = Path(r'\\WARSDV513\aoi\Easy\ET4\Vision Orientation\Failed_images_incoming_ET3')
path_et3_outgoing = Path(r'\\WARSDV513\aoi\Easy\ET4\Vision Orientation\Failed_images_outgoing_ET3')
path_et2 = Path(r'\\WARSDV513\aoi\Easy\ET4\Vision Orientation\Failed_images_incoming_ET2')
path_et2_outgoing = Path(r'\\WARSDV513\aoi\Easy\ET4\Vision Orientation\Failed_images_outgoing_ET2')


In [ ]:
#| export
def get_time_(x):
    return x.split('_')[1][:-2]

In [ ]:
#| export
def get_label_(x):
    if 'Missing' in x:
        return 'Missing'
    elif 'Additional' in x:
        return 'Additional'
    elif 'Flying' in x:
        return 'Flying'
    elif 'DeviceSize' in x:
        return 'Problem with Device Size'
    elif 'Turned Device' in x:
        return 'Turned Device'
    elif 'RefImg' and 'Exception' in x:
        return 'Ref Image Exception'
    else:
        return 'No Label'

In [ ]:
#| export
def get_eqp_name(x):
    'Get eqp name from the file name'
    if 'ET3' in Path(x).parent.name.split('_')[-1]:
        return 'ET3'
    elif 'ET2' in Path(x).parent.name.split('_')[-1]:
        return 'ET2'
    else:
        return 'ET4'

In [ ]:
#| export 
def create_info_df(path):
    #path = Path(path)
    fns = list(map(lambda x: x.as_posix(), path.ls() ))
    df = pd.DataFrame(columns=['path','filename', 'label'])
    df['path'] = fns

    nw_df_=(
        df
        .assign(
            filename = lambda x: x['path'].apply(lambda x: Path(x).name),
            label = lambda x: x['filename'].apply(get_label_),
            date_time_str = lambda x: x['filename'].apply(lambda x: x.split('_')[1][:-2]),
            date_time = lambda x: pd.to_datetime(x['date_time_str'], format='%Y%m%d%H%M%S'),
            eqp_name = lambda x: x['path'].apply(get_eqp_name)
            )
        .loc[:, ['path', 'filename', 'label', 'date_time', 'eqp_name']]
    )
    print(nw_df_['label'].value_counts())
    return nw_df_

In [ ]:
des_path = Path(r'M:\Fail_investigation')
#df_.to_parquet(des_path/'df_20240903.parquet')

# Now the load the parequet files

In [ ]:
df_p = pd.read_parquet(des_path/'df_20240903.parquet')

In [ ]:
df_p.shape

In [ ]:
df_p['eqp_name'].value_counts()

In [ ]:
df_missing_ = df_p.query('label == "Missing"')

In [ ]:
four_weeks_ago = datetime.now() - timedelta(weeks=4)

### Missing 4 weeks ago 

In [ ]:
des_path  = Path(r'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813')
des_path.ls()

In [ ]:
des_path

In [ ]:
df_four_weeks_ago = df_missing_.query('date_time >= @four_weeks_ago')
#df_four_weeks_ago.to_parquet(des_path/'df_four_weeks_ago.parquet')
df_four_weeks_ago.to_parquet(des_path/'last_four_week.parquet')

## Copying the images to the destination folder

In [ ]:
#df_m =pd.read_parquet(des_path/'df_four_weeks_ago.parquet')
df_m_c =pd.read_parquet(des_path/'last_four_week.parquet')

In [ ]:
df_m_c.shape

In [ ]:
df_m.head(), df_m_c

In [ ]:
im_dst = Path(r'M:\Fail_investigation\Missing_lead\all_missing')

In [ ]:
df_m['path'].values[0]

In [ ]:
df_m['path'].values[-1:]

In [ ]:
#for i in tqdm(reversed(df_m['path'].values)):
    #shutil.copyfile(i, Path(im_dst, Path(i).name))

In [ ]:
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
custom_lib_path 

In [ ]:
import sys
sys.path.append(str(custom_lib_path))

In [ ]:
model_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/ModelV12/time_22_18_19_val_frGrnd__0.9645_epoch_118.h5')
im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_2208/missing_2208_unzip/main_im2_cropped_masks/v3/failed/missing/images')

In [ ]:
CURRENT_WD = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection')
env_path = Path(CURRENT_WD, 'private_easy_pin_detection/.env')

In [ ]:
from dotenv import load_dotenv
load_dotenv(env_path)

In [ ]:
CV_TOOLS = os.getenv('CV_TOOLS')
PRIVATE_EASY_PIN_DETECTION = os.getenv('PRIVATE_EASY_PIN_DETECTION')

In [ ]:
sys.path.append(str(CV_TOOLS))
sys.path.append(str(PRIVATE_EASY_PIN_DETECTION))

In [ ]:
#| export 
from cv_tools.core import *
from cv_tools.dataset_check import *

In [ ]:
# Check overlay image

In [ ]:
overlay_mask_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_/missing_2208_unzip/main_im2_cropped_masks/v3/failed/missing/masks')

In [ ]:
overlay_mask_path=Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_0209/missing_0209_unzip/main_im2_cropped_masks/v5/passed/overlay')
move_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_0209/missing_0209_unzip/main_im2_cropped_masks/v5/passed/moved')
move_path.mkdir(exist_ok=True, parents=True)    

In [ ]:
display_image_row(
    im_path=overlay_mask_path,
    move_path=move_path,
    max_images=2,
    start=10,
    im_height=200,
    im_width=200,

)

# Patch image testing

In [ ]:
patch_model_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/PatchModel256/ModelV01')
mdl_name = 'time_23_32_06_val_frGrnd0.9556_epoch_195.h5'
mdl_fn = Path(patch_model_path, mdl_name)
mdl = tf.keras.models.load_model(mdl_fn.as_posix(), compile=False)

In [ ]:
mdl.summary()

In [ ]:
mdl_ = tf.keras.models.load_model(model_path, compile=False)  

In [ ]:
current_dir_path=Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection')
env_path = Path(current_dir_path, 'private_easy_pin_detection/.env')

In [ ]:
sm_fn = im_path.ls()[0]

In [ ]:
img_ = read_img(sm_fn)

In [ ]:
img_b = (img_/255.0)[None, ..., None]
img_b.shape

In [ ]:
show_(mdl_.predict(img_b)[0])

- check the missing pins
- see about rights and start copying

In [ ]:
model_name5 = 'time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5'
model_name = 'time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh9.h5'
base_path = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/Fail_start20240521_unzip/main_im2_cropped_masks/{model_name}/failed/missing/overlay')
t5_path = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/Fail_start20240521_unzip/main_im2_cropped_masks/{model_name5}/passed/overlay')
t5_path_a = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/Fail_start20240521_unzip/main_im2_cropped_masks/{model_name5}/failed/additional/overlay')

In [ ]:
base_path.ls()

# Threshold 9 missing pins

In [ ]:
m_pins = get_name_(base_path.ls())

# Threshold 5 passed images

In [ ]:
pass_ims = get_name_(t5_path.ls())

## checking whether all missing pins are passed or not

In [ ]:
c_m_pins = list(filter(lambda x: x in pass_ims, m_pins))

In [ ]:
len(c_m_pins)

In [ ]:
len(m_pins)

### 2 images difference

## But first check 

In [ ]:
m_pins[0]

In [ ]:
m_pins[idx]

In [ ]:
idx = 5
show_(f'{base_path}/{m_pins[idx]}')

In [ ]:
show_(f'{t5_path_a}/{m_pins[idx]}')

# other situation all `96 models`

In [ ]:
model_name5 = 'time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5'
model_name = 'time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh9.h5'
base_path = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped_masks/{model_name}/failed/missing/overlay')
t5_path = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped_masks/{model_name5}/passed/overlay')
t5_path_a = Path(fr'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped_masks/{model_name5}/failed/additional/overlay')

In [ ]:
m_pins = get_name_(base_path.ls())
pass_ims = get_name_(t5_path.ls())
c_m_pins = list(filter(lambda x: x in pass_ims, m_pins))
len(c_m_pins),len(m_pins)

# Uncommon

In [ ]:
n_mp = list(filter(lambda x: x not in c_m_pins, m_pins ))

In [ ]:
idx = 5
show_(Path(base_path, n_mp[idx]))

# index 5 is intersting

- But we can cut `37` images pin and create for mask for them

# Check missing pin issue in Single and double pin image

In [ ]:
data_path = Path(os.getenv('TEST_DATA_PATH'), 'missing_test_unzip')
im1_path = Path(data_path, 'main_im1_cropped')
im2_path = Path(data_path, 'main_im2_cropped')
pr_im_path = Path(data_path, 'main_pr_cropped')
lib_path = Path(current_dir_path, 'private_easy_pin_detection')
config_sn_img = Path(lib_path, 'config_office_test.yaml')
config_two_img = Path(lib_path, 'config_test_two_image.yml')

In [ ]:
def read_config(config_sn_img:str)->dict:
    with open(config_sn_img, 'r') as file:
        config = yaml.safe_load(file)
    return config

In [ ]:
config_ = read_config(config_sn_img)
config_t = read_config(config_two_img)

In [ ]:
model_two_img = load_model(config_t)

In [ ]:
stack_img = process_image_im1_im2_j(
    im1=im1_img,
    im2=im2_img
)

In [ ]:
model_inp = tf.transpose(stack_img, (0,3,1,2))

In [ ]:
predictions = model_two_img.predict(model_inp)

In [ ]:
show_(predictions[0,:, :, 0])

In [ ]:
def one_image_pred(config, im1_img, show=True):
    model = load_model(config)
    pred = tf.cast(im1_img[None,...]/255.0, tf.float32)[..., None]
    print(pred.shape)
    pred = model.predict(pred)
    pred = pred > 0.5
    if show:show_(pred[0,:, :, 0])
    return pred[0,:, :, 0]

In [ ]:
model = load_model(config_)

In [ ]:
img = read_normalize(im_file=f'{fn}', config=config_)[None, ...]
img.shape

In [ ]:
msk = (model.predict(img) > 0.5)[0,:, :, 0]

In [ ]:
show_(msk)

In [ ]:
im1_pr = tf.cast((tf.cast(im1_img, tf.float32)/255.0) > 0.3, tf.float32)

In [ ]:
m_img = im1_pr * msk
show_(m_img)

In [ ]:
show_((model.predict(img)>0.5)[0,...,0])

In [ ]:
im2_path.ls()

In [ ]:
fn = im2_path.ls()[0]
im2_img = read_img(f'{fn}')
im1_name = Path(f"{Path(fn).name.replace('image2', 'image1')}")
im1_img = read_img(Path(im1_path, im1_name))
pr_name = Path(f"{Path(fn).name.replace('image2', 'processed_image')}")
pr_img = read_img(Path(pr_im_path, pr_name), gray=False)

In [ ]:
show_(pr_img)

In [ ]:
from private_easy_pin_detection.mask_generation_two_image import *

# In case of additional pin problem

In [ ]:
a_data_path = Path(r'/home/ai_sintercra.work/Fail_investigation/additional/two_image_test/')
a_im1_path = Path(a_data_path, 'main_im1_cropped')
a_im2_path = Path(a_data_path, 'main_im2_cropped')
a_pr_path = Path(a_data_path, 'main_pr_cropped')

In [ ]:
idx = 2
a_im2_fn = a_im2_path.ls()[idx]
a_fn_  = Path(a_im2_fn).name
a_im2_fn = Path(a_im2_path, a_fn_)
a_fn1 = a_fn_.replace('image2', 'image1')
a_im1_fn = Path(a_im1_path, a_fn1)
a_pr_fn = a_fn_.replace('image2', 'processed_image')
a_pr_fn = Path(a_pr_path, a_pr_fn)

In [ ]:
msk = one_image_pred(config_, read_img(f'{a_im2_fn}'), show=True)

In [ ]:
im1_img_ = read_img(a_im1_fn)

In [ ]:
im1_img_t = (im1_img_/255.0) > 0.2

In [ ]:
show_(msk * im1_img_t)

In [ ]:
cntrs_b = find_contours_binary(msk.astype(np.uint8))

In [ ]:
x, y, w, h

In [ ]:
cntrs_

In [ ]:
msk_img = msk.astype(np.uint8)

In [ ]:
idx_ = 0 
x, y, w, h = cntrs_[idx_]
offset=10
show_(msk_img[y-offset:y+h+offset, x-offset:x+w+offset])

In [ ]:
add_msk = msk_img[y-offset:y+h+offset, x-offset:x+w+offset]
add_im1_part = im1_nt[y-offset:y+h+offset, x-offset:x+w+offset]

In [ ]:
show_(add_im1_part)

In [ ]:
add_msk_ = add_msk.astype(bool)
add_im1_part_ = add_im1_part.numpy().astype(bool)

In [ ]:
im1_nt = tf.cast(tf.cast((a_im1_img /255.0), tf.float32) > 0.1, tf.float32)
im1_nt.shape

In [ ]:
show_(im1_nt[y-offset:y+h+offset, x-offset:x+w+offset])

In [ ]:
show_(im1_nt)

In [ ]:
show_(msk * im1_nt)